## 0. Setup

In [25]:
# imports

import os, io, json, warnings, gc, shutil, re, platform
from copy import deepcopy
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy.ndimage import zoom, gaussian_filter

import s3fs
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from monai.utils import set_determinism
from monai.networks.nets import DenseNet121

from sklearn.model_selection import train_test_split

In [2]:
# seeds
SEED = 42
set_determinism(seed=SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
# outputs
WORK_DIR    = Path.home() / "tms_fmri_project"
MAPS_DIR    = WORK_DIR / "response_maps"  
TEMP_DIR    = WORK_DIR / "temp_nifti"      # raw NIfTIs deleted after use
RESULTS_DIR = WORK_DIR / "results"
for d in [WORK_DIR, MAPS_DIR, TEMP_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [4]:
# dataset
DATASET_ID = "ds005498"
S3_BUCKET  = "openneuro.org"

# acquisition parameters (from paper)
TR_TMS       = 2.4   # seconds
HRF_OFFSET   = 2     # volumes after pulse before averaging
HRF_WIN      = 3     # volumes to average (HRF peak ~5–7 s post-stim)
BASELINE_WIN = 2     # volumes before pulse for baseline

# cohort labels
GROUPS    = ["NTHC", "TEHC", "NTS", "NIS"]
LABEL_MAP = {g: i for i, g in enumerate(GROUPS)}

In [5]:
# NVIDIA QUADRO P520 SPECIFIC
TARGET_SHAPE = (48, 48, 36)

In [6]:
# hyperparameters
BATCH_SIZE   = 2     # should be safe for 2 GB vram
N_EPOCHS     = 60
LR           = 1e-4
WEIGHT_DECAY = 1e-5
N_FOLDS      = 5
DROPOUT      = 0.5
PATIENCE     = 15
TEST_FRAC    = 0.20

In [ ]:
NUM_WORKERS = 0 if platform.system() == "Windows" else 4

#note: might not need device param
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {DEVICE}")
print(f"TARGET_SHAPE : {TARGET_SHAPE}")
print(f"BATCH_SIZE   : {BATCH_SIZE}")
print(f"Work dir     : {WORK_DIR}")

Device       : cuda
TARGET_SHAPE : (48, 48, 36)
BATCH_SIZE   : 2
Work dir     : C:\Users\jules\tms_fmri_project


## 1. Deal With Data


### 1.1 S3 Functions
For dealing with downloading and deleting files.

In [11]:
def get_s3() -> s3fs.S3FileSystem:
    return s3fs.S3FileSystem(anon=True)

def s3_ls(fs, prefix: str) -> list:
    try:
        return fs.ls(prefix, detail=False)
    except Exception as e:
        print(f"  [s3_ls] {prefix}: {e}")
        return []

def s3_read_tsv(fs, path: str) -> pd.DataFrame:
    with fs.open(path, "rb") as f:
        return pd.read_csv(f, sep="\t", low_memory=False)

def s3_download(fs, s3_path: str, local_path: Path):
    with fs.open(s3_path, "rb") as src, open(local_path, "wb") as dst:
        shutil.copyfileobj(src, dst)

def extract_task(fname: str):
    m = re.search(r"task-([^_]+)", Path(fname).name)
    return m.group(1) if m else None

fs  = get_s3()
pfx = f"{S3_BUCKET}/{DATASET_ID}"
top = s3_ls(fs, pfx)
print(f"✓ Connected — top-level entries under s3://{pfx}: {len(top)}")


✓ Connected — top-level entries under s3://openneuro.org/ds005498: 158


### 1.2 Load and Clean Data

In [12]:
participants_df = s3_read_tsv(fs, f"{pfx}/participants.tsv")
print(f"Raw shape: {participants_df.shape}")
print(participants_df.head(3))

# get group from id string (e.g. sub-NTHC001 → NTHC)
def get_group_from_id(sid):
    s = str(sid).upper()
    if "NTHC" in s: return "NTHC"
    if "TEHC" in s: return "TEHC"
    if "NTS"  in s: return "NTS"
    if "NIS"  in s or "TIS" in s: return "NIS"
    return None

participants_df["group"] = participants_df["participant_id"].apply(get_group_from_id)

# standardize sex column name
for col in participants_df.columns:
    if col.lower().strip() in ("sex", "gender", "sex_at_birth", "biological_sex"):
        participants_df = participants_df.rename(columns={col: "sex"})
        break

#  sex as [0, 1]
# BIDS convention: 1=Female, 2=Male 
SEX_NORM = {
    "M": 1.0, "m": 1.0, "male": 1.0, "Male": 1.0,
    "2": 1.0, 2: 1.0,
    "F": 0.0, "f": 0.0, "female": 0.0, "Female": 0.0,
    "1": 0.0, 1: 0.0,
}
participants_df["sex_num"] = participants_df["sex"].map(SEX_NORM).fillna(0.5)

# int labels and clean
participants_df["label"]   = participants_df["group"].map(LABEL_MAP)
participants_df            = participants_df[participants_df["label"].notna()].copy()
participants_df            = participants_df.rename(columns={"participant_id": "subject_id"})
participants_df["label"]   = participants_df["label"].astype(int)
participants_df            = participants_df.reset_index(drop=True)

print("\nGroup distribution:")
print(participants_df["group"].value_counts())
print("\nSex numeric distribution:")
print(participants_df["sex_num"].value_counts())

Raw shape: (152, 51)
  participant_id  Gender  Race  Ethnicity  Age  Years of Education  \
0   sub-NTHC1001       2     5          2   45                  18   
1   sub-NTHC1003       2     4          2   24                  16   
2   sub-NTHC1009       2     2          2   42                  14   

   PHQ-9 Score  GAD-7 Score  PCL-5 Score  Last Month CAPS-5  ...  \
0          0.0          NaN          NaN                NaN  ...   
1          0.0          NaN          NaN                NaN  ...   
2          0.0          0.0          NaN                NaN  ...   

   R-FEF Scan (34x6x62)  R-IFJ Scan (50x10x28)  R-IPL Scan (48xMinus54x46)  \
0                     0                      0                           0   
1                     0                      1                           0   
2                     1                      0                           0   

   R-M1 Scan (40xMinus18x64)  R-preSMA Scan (6x2x70)  R-aMFG Scan (30x50x26)  \
0                          0    

### 1.3 80/20 train/test split

In [13]:
all_sids = participants_df["subject_id"].values
all_labs = participants_df["label"].values

TRAIN_SIDS, TEST_SIDS, _, _ = train_test_split(
    all_sids, all_labs,
    test_size=TEST_FRAC,
    stratify=all_labs,
    random_state=SEED,
)
TRAIN_SIDS, TEST_SIDS = list(TRAIN_SIDS), list(TEST_SIDS)

print(f"Total     : {len(all_sids)}")
print(f"Train     : {len(TRAIN_SIDS)}  ({100*(1-TEST_FRAC):.0f}%)")
print(f"Test      : {len(TEST_SIDS)}   ({100*TEST_FRAC:.0f}%)")
print()
print("Train distribution:")
print(participants_df[participants_df["subject_id"].isin(TRAIN_SIDS)]["group"].value_counts().to_string())
print()
print("Test distribution:")
print(participants_df[participants_df["subject_id"].isin(TEST_SIDS)]["group"].value_counts().to_string())


Total     : 152
Train     : 121  (80%)
Test      : 31   (20%)

Train distribution:
group
NTHC    37
NTS     34
NIS     27
TEHC    23

Test distribution:
group
NTHC    9
NTS     9
NIS     7
TEHC    6


### 1.4 Get TMS Task Names & Isolate Causal BOLD Signal
Helping convert from 4D to 3D data.

In [ ]:
# --- getting the TMS task names ---

def get_subject_func_files(fs, pfx, subject_id):
    files = []
    files.extend(s3_ls(fs, f"{pfx}/{subject_id}/func"))
    sub_items = s3_ls(fs, f"{pfx}/{subject_id}")
    for item in sub_items:
        if "ses-" in item.split("/")[-1]:
            files.extend(s3_ls(fs, f"{item}/func"))
    return files

def discover_all_tms_tasks(fs, pfx, subject_ids, max_search=20):
    all_tasks = set()
    for sid in subject_ids[:max_search]:
        files = get_subject_func_files(fs, pfx, sid)
        bold = [f for f in files if "bold.nii" in f]
        for f in bold:
            task = extract_task(f)
            if task and "rest" not in task.lower():
                all_tasks.add(task)
        # break early if we found all 11 sites mentioned in the paper
        if len(all_tasks) == 11:
            break
    return sorted(list(all_tasks))

# use the new function to scan multiple subjects instead of just one
all_subject_ids = participants_df["subject_id"].tolist()
TMS_TASKS = discover_all_tms_tasks(fs, pfx, all_subject_ids)

TASK_TO_IDX      = {t: i for i, t in enumerate(TMS_TASKS)}
N_SITES          = len(TASK_TO_IDX)
SITE_IDX_TO_TASK = {v: k for k, v in TASK_TO_IDX.items()}

print(f"\nTMS site -> channel index ({N_SITES} sites):")
for t, i in TASK_TO_IDX.items():
    print(f"  [{i:2d}] {t}")

  [s3_ls] openneuro.org/ds005498/sub-NTHC1001/func: openneuro.org/ds005498/sub-NTHC1001/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1003/func: openneuro.org/ds005498/sub-NTHC1003/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1009/func: openneuro.org/ds005498/sub-NTHC1009/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1015/func: openneuro.org/ds005498/sub-NTHC1015/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1015/ses-2/func: openneuro.org/ds005498/sub-NTHC1015/ses-2/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1016/func: openneuro.org/ds005498/sub-NTHC1016/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1019/func: openneuro.org/ds005498/sub-NTHC1019/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1021/func: openneuro.org/ds005498/sub-NTHC1021/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1022/func: openneuro.org/ds005498/sub-NTHC1022/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1023/func: openneuro.org/ds005498/sub-NTHC1023/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1024/func: openne

### 1.5 TMS Response Map Computation

In [ ]:

def load_pulse_onsets(fs, events_s3: str) -> np.ndarray:
    try:
        ev = s3_read_tsv(fs, events_s3)
        if "onset" in ev.columns:
            return ev["onset"].dropna().values.astype(float)
    except Exception:
        pass
    return np.array([])


def compute_response_map(
    bold_4d: np.ndarray,
    onsets: np.ndarray,
    tr: float = TR_TMS,
    hrf_offset: int = HRF_OFFSET,
    hrf_win: int = HRF_WIN,
    base_win: int = BASELINE_WIN,
) -> np.ndarray:
    """Voxelwise average TMS response map: (post-stim mean) - (pre-stim baseline)."""
    T = bold_4d.shape[3]
    post_maps, base_maps = [], []
    for t0 in onsets:
        v0     = int(t0 / tr)
        v_post = v0 + hrf_offset
        v_end  = v_post + hrf_win
        v_base = max(0, v0 - base_win)
        if v_end <= T and v0 > v_base:
            post_maps.append(bold_4d[..., v_post:v_end].mean(axis=3))
            base_maps.append(bold_4d[..., v_base:v0].mean(axis=3))
    if not post_maps:
        return np.zeros(bold_4d.shape[:3], dtype=np.float32)
    return (np.stack(post_maps).mean(0) - np.stack(base_maps).mean(0)).astype(np.float32)


def resample_volume(vol: np.ndarray, target: tuple = TARGET_SHAPE) -> np.ndarray:
    factors = tuple(t / s for t, s in zip(target, vol.shape))
    return gaussian_filter(zoom(vol, factors, order=1, prefilter=False), sigma=0.5).astype(np.float32)


def z_score(vol: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    return ((vol - vol.mean()) / (vol.std() + eps)).astype(np.float32)


### 1.6 Data Preprocessing

In [21]:
MAX_SUBJECTS = None   # set to e.g. 10 for a quick test

def process_subject(fs, pfx, subject_id, maps_dir, temp_dir):
    sub_dir = maps_dir / subject_id
    sub_dir.mkdir(exist_ok=True)

    # use the helper function here instead of s3_ls directly
    files = get_subject_func_files(fs, pfx, subject_id)
    
    bold_by_task = {
        extract_task(f): f
        for f in files
        if "bold.nii" in f and extract_task(f) in TASK_TO_IDX
    }

    # the rest of the function stays exactly the same
    results = {}
    for task, bold_s3 in bold_by_task.items():
        site_idx = TASK_TO_IDX[task]
        npy_path = sub_dir / f"site_{site_idx:02d}.npy"

        if npy_path.exists():
            results[site_idx] = npy_path
            continue

        events_s3 = re.sub(r"bold\.nii(\.gz)?$", "events.tsv", bold_s3)
        temp_nii  = temp_dir / f"{subject_id}_{task}.nii.gz"
        try:
            s3_download(fs, bold_s3, temp_nii)
            bold_img  = nib.load(str(temp_nii))
            bold_data = bold_img.get_fdata(dtype=np.float32)
            onsets    = load_pulse_onsets(fs, events_s3)
            if len(onsets) == 0:
                n_vols = bold_data.shape[3]
                onsets = np.linspace(TR_TMS * 4, n_vols * TR_TMS - TR_TMS * 4, 68)
                print(f"      warn {task}: no events.tsv, using estimated timings")
            resp = compute_response_map(bold_data, onsets)
            resp = resample_volume(resp, TARGET_SHAPE)
            resp = z_score(resp)
            np.save(str(npy_path), resp)
            results[site_idx] = npy_path
            del bold_img, bold_data, resp
            gc.collect()
        except Exception as e:
            print(f"      error {subject_id}/{task}: {e}")
        finally:
            if temp_nii.exists():
                temp_nii.unlink()
    return results

subject_ids = participants_df["subject_id"].tolist()
if MAX_SUBJECTS:
    subject_ids = subject_ids[:MAX_SUBJECTS]

print(f"Preprocessing {len(subject_ids)} subjects (skipping already-done) …\n")
SUBJECT_MAPS: dict = {}

for sid in tqdm(subject_ids, desc="Subjects"):
    SUBJECT_MAPS[sid] = process_subject(fs, pfx, sid, MAPS_DIR, TEMP_DIR)

done      = sum(1 for v in SUBJECT_MAPS.values() if v)
site_cnts = [len(v) for v in SUBJECT_MAPS.values() if v]
print(f"\n✅ Done. {done} subjects with ≥1 map.")
if site_cnts:
    print(f"   Sites per subject — mean {np.mean(site_cnts):.1f}, min {min(site_cnts)}, max {max(site_cnts)}")


Preprocessing 152 subjects (skipping already-done) …



Subjects:   0%|          | 0/152 [00:00<?, ?it/s]

  [s3_ls] openneuro.org/ds005498/sub-NTHC1001/func: openneuro.org/ds005498/sub-NTHC1001/func
      warn stim46x26x38: no events.tsv, using estimated timings
  [s3_ls] openneuro.org/ds005498/sub-NTHC1003/func: openneuro.org/ds005498/sub-NTHC1003/func
      warn stim50x10x28: no events.tsv, using estimated timings
  [s3_ls] openneuro.org/ds005498/sub-NTHC1009/func: openneuro.org/ds005498/sub-NTHC1009/func
      warn stim30x50x26: no events.tsv, using estimated timings
      warn stim34x6x62: no events.tsv, using estimated timings
      warn stim40xMinus18x64: no events.tsv, using estimated timings
      warn stim46x26x38: no events.tsv, using estimated timings
      warn stimMinus32x42x34: no events.tsv, using estimated timings
      warn stimMinus38x22x48: no events.tsv, using estimated timings
  [s3_ls] openneuro.org/ds005498/sub-NTHC1015/func: openneuro.org/ds005498/sub-NTHC1015/func
  [s3_ls] openneuro.org/ds005498/sub-NTHC1015/ses-2/func: openneuro.org/ds005498/sub-NTHC1015/ses-2/fu

### 1.7 Dataset Class for Pytorch

In [28]:
# --- dataset and loader ---

class SimpleTMSDataset(Dataset):
    def __init__(self, subject_ids, subject_maps, meta_df):
        self.sids = subject_ids
        self.maps = subject_maps
        self.meta = meta_df.set_index("subject_id")
        
    def __len__(self):
        return len(self.sids)
        
    def __getitem__(self, idx):
        sid = self.sids[idx]
        info = self.meta.loc[sid]
        
        # empty volume for all possible sites
        volume = np.zeros((N_SITES, *TARGET_SHAPE), dtype=np.float32)
        
        # fill in the channels for the sites this person actually completed
        for si, path in self.maps.get(sid, {}).items():
            if si < N_SITES:
                volume[si] = np.load(str(path))
                
        label = int(info["label"])
        
        return {
            "volume": torch.from_numpy(volume),
            "label": torch.tensor(label, dtype=torch.long)
        }

# only used successfully downloaded subjects with maps
train_sids_ready = [s for s in TRAIN_SIDS if s in SUBJECT_MAPS and SUBJECT_MAPS[s]]
test_sids_ready = [s for s in TEST_SIDS if s in SUBJECT_MAPS and SUBJECT_MAPS[s]]

train_ds = SimpleTMSDataset(train_sids_ready, SUBJECT_MAPS, participants_df)
test_ds = SimpleTMSDataset(test_sids_ready, SUBJECT_MAPS, participants_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Ready to train with {len(train_ds)} train subjects and {len(test_ds)} test subjects")

Ready to train with 121 train subjects and 31 test subjects


## 2. Model Setup

In [29]:
# init MONAI 3D classification model
model = DenseNet121(
    spatial_dims=3, 
    in_channels=N_SITES,  # 11 channels, one for each stimulation site
    out_channels=4        # 4 clinical groups (NTHC, TEHC, NTS, NIS)
).to(DEVICE)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Model, loss function, and optimizer initialized.")

Model, loss function, and optimizer initialized.


## 3. Training Loop

In [30]:
print("Starting training...")

best_acc = 0

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    
    # --- Training Step ---
    for batch in train_loader:
        inputs = batch["volume"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    # --- Evaluation Step ---
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch["volume"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    test_acc = correct / total
    
    # print update every 5 epochs / on first epoch
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{N_EPOCHS} - Train Loss: {epoch_loss/len(train_loader):.4f} - Test Acc: {test_acc:.4f}")
    
    # save best version of model
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), RESULTS_DIR / "best_model.pth")

print(f"\nFinished! Best test accuracy was {best_acc:.4f}")
print(f"Best model weights saved to: {RESULTS_DIR / 'best_model.pth'}")

Starting training...
Epoch 01/60 - Train Loss: 1.4279 - Test Acc: 0.3226
Epoch 05/60 - Train Loss: 1.3983 - Test Acc: 0.1613
Epoch 10/60 - Train Loss: 1.3975 - Test Acc: 0.3548
Epoch 15/60 - Train Loss: 1.3449 - Test Acc: 0.2581
Epoch 20/60 - Train Loss: 1.3659 - Test Acc: 0.1613
Epoch 25/60 - Train Loss: 1.3600 - Test Acc: 0.1935
Epoch 30/60 - Train Loss: 1.3885 - Test Acc: 0.2581
Epoch 35/60 - Train Loss: 1.3791 - Test Acc: 0.2903
Epoch 40/60 - Train Loss: 1.3753 - Test Acc: 0.2581
Epoch 45/60 - Train Loss: 1.3915 - Test Acc: 0.3871
Epoch 50/60 - Train Loss: 1.3778 - Test Acc: 0.2903
Epoch 55/60 - Train Loss: 1.3844 - Test Acc: 0.3226
Epoch 60/60 - Train Loss: 1.3500 - Test Acc: 0.2258

Finished! Best test accuracy was 0.3871
Best model weights saved to: C:\Users\jules\tms_fmri_project\results\best_model.pth


notes: try doing visualization of what data looks like before going into model

1. try higher resolution with more compute -- email mark and maged
2. 1.35 is pretty high train loss (still underfit) do more training, plot accuracy and loss (overfit if loss increase while acc decrease)
3. add in data vis - show examples, class averages
4. also print confusion matrix (which classes are correct and which commonly are mixed up)
5. see if there's augmentations people tend to use in lit for other fmri classification i.e. adding guassian noise to improve generalization

exploratory justification--discovering new biomarker
interpretability methods --> gradcam (python packages exist for this) will show you which region of the image it's using

during testing - split cases into categories and do stimulation site specific analysis
mini model idea could make dataset really small -- maybe just split sites for testing